In [ ]:
import glob
import json
import requests
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import time

def fetch_wikipedia_summary(label, session):
    """Fetch the first two paragraphs from Wikipedia for the given label."""
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{label}"
    response = session.get(url)
    time.sleep(0.1)  # Adding a small delay to prevent being rate limited

    if response.status_code != 200:
        summary = f"Page for '{label}' does not exist on Wikipedia.\n\n"
    else:
        data = response.json()
        if 'extract' in data:
            summary_paragraphs = data['extract'].split('\n\n')[:2]
            summary = "\n\n".join(summary_paragraphs)
        else:
            summary = f"No summary found for '{label}'.\n\n"
    
    return label, summary

def process_jsonl_file(file):
    """Process a single JSONL file and fetch Wikipedia summaries based on labels."""
    user_agent = 'YourUserAgent/1.0 (yourname@example.com)'
    session = requests.Session()
    session.headers.update({'User-Agent': user_agent})
    
    labels = []
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line.strip())
                label = record.get('label')

                if not label:
                    continue

                labels.append(label)
            except json.JSONDecodeError:
                print(f"Error decoding JSON in file: {file}")
            except Exception as e:
                print(f"Unexpected error processing line in file {file}: {line}. Error: {e}")
    
    results = []
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(fetch_wikipedia_summary, label, session): label for label in labels}
        for future in as_completed(futures):
            label, summary = future.result()
            results.append((label, summary))
    
    return results

def main():
    # Get all JSONL file paths
    records_files = glob.glob('labels/*.jsonl')

    # Output file to save summaries
    output_file = 'wikipedia_summaries.txt'

    with open(output_file, 'w', encoding='utf-8') as out_f:
        with ProcessPoolExecutor(max_workers=76) as executor:
            futures = {executor.submit(process_jsonl_file, file): file for file in records_files}
            
            for future in tqdm(as_completed(futures), total=len(futures), desc="Processing files", unit="file"):
                file = futures[future]
                try:
                    results = future.result()
                    for label, summary in results:
                        out_f.write(f"Label: {label}\n")
                        out_f.write(f"Summary:\n{summary}\n")
                        out_f.write("="*80 + "\n")
                except Exception as e:
                    print(f"Error processing file {file}. Error: {e}")

    print(f"Wikipedia summaries saved to {output_file}")

if __name__ == "__main__":
    main()
